# Master Orchestrator
This notebook acts as the master driver for all 104 comparative model notebooks in `Paper 1`.

It will automatically locate all `.ipynb` files (except itself) and execute them in parallel using your 96GB RTX 6000 Pro.
Each notebook will save its results and outputs *back into itself*, so you can inspect them later.

In [ ]:
import os
import glob
import papermill as pm
import concurrent.futures
from tqdm.notebook import tqdm

paper_dir = 'Paper 1'
if not os.path.exists(paper_dir):
    print(f'Error: Could not find {paper_dir} directory.')
all_notebooks = glob.glob(f'{paper_dir}/**/*.ipynb', recursive=True)
all_notebooks = [nb for nb in all_notebooks if 'Master_Runner' not in nb and '.ipynb_checkpoints' not in nb]

print(f'Found {len(all_notebooks)} notebooks to execute.')

# 2. Define the execution function
def execute_notebook(nb_path):
    try:
        # Execute the notebook and save the output back to itself
        pm.execute_notebook(
            input_path=nb_path,
            output_path=nb_path,
            cwd=paper_dir, # Force CWD to Paper 1 so 'content/dataset.csv' paths work flawlessly
            log_output=False
        )
        return (nb_path, True, None)
    except Exception as e:
        return (nb_path, False, str(e))

# 3. Execute in parallel
# Safe concurrency for 96GB VRAM. Adjust max_workers if you run out of memory.
MAX_WORKERS = 10

results = []
print(f'Starting execution with {MAX_WORKERS} parallel workers...')

with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(execute_notebook, nb) for nb in all_notebooks]
    
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(all_notebooks), desc="Running Models"):
        nb_path, success, error_msg = future.result()
        if success:
            results.append(nb_path)
        else:
            print(f'❌ Failed: {nb_path} | Error: {error_msg}')

print(f'\n✅ Execution Complete! Successfully ran {len(results)} out of {len(all_notebooks)} notebooks.')
